# Sprint 2: Model Building & Evaluation
Hotel Booking Cancellation Prediction. Upload `hotel_bookings.csv` to Colab/Jupyter, then run all cells.


In [ ]:
!pip -q install pandas numpy scikit-learn joblib


In [ ]:
DATA_PATH = 'hotel_bookings.csv'


## Train and compare six classification models
The code below repeats the Sprint 1 cleaning and uses a stratified 80:20 split. All preprocessing is fitted on training data only.


In [ ]:
"""
SPRINT 2 - Hotel Booking Cancellation Prediction: Model Building & Evaluation
Install:
    pip install pandas numpy scikit-learn joblib matplotlib seaborn
Run:
    python sprint2_model_comparison.py --data hotel_bookings.csv

This script reproduces Sprint 1 cleaning and Sprint 2 model comparison.
"""
import argparse
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

warnings.filterwarnings("ignore")

def main(data_path):
    out = Path("sprint2_output")
    out.mkdir(exist_ok=True)

    # Reproduce Sprint 1 cleaning
    df = pd.read_csv(data_path)
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_"))
    duplicates = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    df["children"] = df["children"].fillna(df["children"].median())
    df["country"] = df["country"].fillna("Unknown")
    for col in ["agent", "company"]:
        df[col] = df[col].fillna("Unknown").astype(str)
    valid_adr_median = df.loc[df["adr"] >= 0, "adr"].median()
    df.loc[df["adr"] < 0, "adr"] = valid_adr_median
    df["total_guests"] = df[["adults", "children", "babies"]].sum(axis=1)
    df["total_stay_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    for col in ["lead_time", "adr", "days_in_waiting_list"]:
        df[col] = np.log1p(df[col].clip(lower=0))

    # Prevent target leakage: these columns reveal or follow the final booking outcome.
    leakage_cols = [c for c in
        ["reservation_status", "reservation_status_date", "assigned_room_type"]
        if c in df.columns]
    X = df.drop(columns=["is_canceled"] + leakage_cols)
    y = df["is_canceled"].astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )
    categorical = X.select_dtypes(include=["object"]).columns.tolist()
    numerical = X.select_dtypes(exclude=["object"]).columns.tolist()

    # One-hot encoding for linear models; scaling for numerical variables.
    onehot = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numerical),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encode", OneHotEncoder(handle_unknown="ignore", min_frequency=10))
        ]), categorical)
    ])

    # Ordinal encoding for tree models and Gaussian Naive Bayes.
    ordinal = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numerical),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encode", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), categorical)
    ])

    models = {
        "Logistic Regression": Pipeline([
            ("prep", onehot),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced",
                                         solver="liblinear"))
        ]),
        "Decision Tree": Pipeline([
            ("prep", ordinal),
            ("model", DecisionTreeClassifier(max_depth=12, min_samples_leaf=20,
                                             class_weight="balanced", random_state=42))
        ]),
        "Random Forest": Pipeline([
            ("prep", ordinal),
            ("model", RandomForestClassifier(n_estimators=160, max_depth=18,
                min_samples_leaf=5, class_weight="balanced_subsample",
                n_jobs=-1, random_state=42))
        ]),
        "Gradient Boosting": Pipeline([
            ("prep", ordinal),
            ("model", GradientBoostingClassifier(n_estimators=100,
                learning_rate=0.08, max_depth=3, random_state=42))
        ]),
        "SVM (Linear)": Pipeline([
            ("prep", onehot),
            ("model", LinearSVC(class_weight="balanced", random_state=42,
                                max_iter=4000))
        ]),
        "Naive Bayes": Pipeline([
            ("prep", ordinal),
            ("model", GaussianNB())
        ])
    }

    rows = []
    fitted = {}
    for name, pipeline in models.items():
        pipeline.fit(X_train, y_train)
        fitted[name] = pipeline
        train_pred = pipeline.predict(X_train)
        test_pred = pipeline.predict(X_test)
        train_acc = accuracy_score(y_train, train_pred)
        test_acc = accuracy_score(y_test, test_pred)
        if hasattr(pipeline, "predict_proba"):
            score = pipeline.predict_proba(X_test)[:, 1]
        else:
            score = pipeline.decision_function(X_test)
        row = {
            "Model": name,
            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,
            "Precision": precision_score(y_test, test_pred, zero_division=0),
            "Recall": recall_score(y_test, test_pred, zero_division=0),
            "F1 Score": f1_score(y_test, test_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, score),
            "Accuracy Gap": train_acc - test_acc,
            "Confusion Matrix": str(confusion_matrix(y_test, test_pred).tolist())
        }
        rows.append(row)
        print("\\n", name)
        print(classification_report(y_test, test_pred, zero_division=0))
        print("Confusion matrix:\\n", confusion_matrix(y_test, test_pred))

    results = pd.DataFrame(rows).sort_values("F1 Score", ascending=False)
    results.to_csv(out / "model_comparison.csv", index=False)
    results.drop(columns=["Confusion Matrix"]).to_csv(out / "model_metrics.csv", index=False)
    print("\\nMODEL COMPARISON")
    print(results.to_string(index=False))

    # Save selected model pipeline (highest test F1-score).
    best_name = results.iloc[0]["Model"]
    joblib.dump(fitted[best_name], out / "best_hotel_cancellation_model.joblib")
    (out / "best_model.txt").write_text(
        f"Selected by highest test F1-score: {best_name}\\n"
        "Review precision, recall and business costs before deployment.\\n",
        encoding="utf-8"
    )
    print(f"\\nBest model by F1-score: {best_name}")
    print(f"Removed exact duplicates: {duplicates}")
    print(f"Clean dataset shape: {df.shape}")
    print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)}")
    print(f"Excluded leakage columns: {leakage_cols}")
    print(f"Results and model saved in: {out.resolve()}")

main(DATA_PATH)
